# 06 · Transformer sobre el texto (corre en Kaggle con GPU)

Fine-tuning de BETO (BERT en español) para predecir nuevo/usado a partir de la **ficha completa** de la publicación escrita como texto: título, precio, stock, tipo de publicación, envío, medios de pago, etiquetas, perfil del vendedor (calculado sobre train+test, sin etiquetas) y garantía. A diferencia de la versión que solo lee el título, puede aprender combinaciones como "este producto a este precio con stock 1", que es lo que distingue a los vendedores sin historial.
No se usa como modelo final sino como **feature de nivel 1** para los modelos de árboles, igual que el TF-IDF.

Para que la validación no tenga fuga de información se respeta el mismo esquema anidado del resto del trabajo:
por cada fold externo `f` (mismos folds que `features.cv_folds`) se entrena

- 5 modelos internos sobre la parte de entrenamiento del fold, para obtener su predicción *out-of-fold*;
- 1 modelo con toda la parte de entrenamiento, que predice la parte de validación y el test.

Son 30 entrenamientos cortos. La salida es `bert_level1.npz`.

In [ ]:
import glob, os, time, re, unicodedata, numpy as np, pandas as pd, torch
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, logging
logging.set_verbosity_error()

MODEL, MAXLEN, EPOCHS, BS, LR = "dccuchile/bert-base-spanish-wwm-uncased", 128, 3, 64, 3e-5
DEBUG = int(os.environ.get("DEBUG_ROWS", 0))          # solo para probar el codigo en local
dev = "cuda" if torch.cuda.is_available() else "cpu"
print(dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(name) + glob.glob(f"../{name}")
    return hits[0]

def norm_text(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s.lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip()

cols = ["title", "warranty"]
train = pd.read_json(find("train_data.jsonlines"), lines=True, convert_dates=False)
test = pd.read_json(find("test_data.jsonlines"), lines=True, convert_dates=False)
if DEBUG:
    train, test = train.iloc[:DEBUG], test.iloc[:DEBUG // 2]
y = (train["condition"] == "new").astype(int).values
# perfil del vendedor sobre train+test (no usa la etiqueta)
both = pd.concat([train.drop(columns=["condition"]), test], ignore_index=True)
g = both.groupby("seller_id")
both["s_n"] = g["id"].transform("size")
both["s_iq1"] = g["initial_quantity"].transform(lambda v: (v == 1).mean())
both["s_free"] = g["listing_type_id"].transform(lambda v: (v == "free").mean())

def ficha(r):
    sh = r["shipping"] if isinstance(r["shipping"], dict) else {}
    pays = ", ".join(p["description"] for p in r["non_mercado_pago_payment_methods"]) or "solo mercadopago"
    si = lambda b: "si" if b else "no"
    return " | ".join([
        str(r["title"]),
        f"precio {r['price']:.0f} {r['currency_id']}",
        f"stock {int(r['initial_quantity'])} vendidos {int(r['sold_quantity'])} disponibles {int(r['available_quantity'])}",
        f"publicacion {r['listing_type_id']} {r['buying_mode']}",
        f"vendedor {int(r['s_n'])} publicaciones, {100 * r['s_iq1']:.0f}% con stock 1, {100 * r['s_free']:.0f}% gratuitas",
        f"envio {sh.get('mode')} gratis {si(sh.get('free_shipping'))} retiro {si(sh.get('local_pick_up'))}",
        f"fotos {len(r['pictures'])} video {si(isinstance(r['video_id'], str) and r['video_id'].strip())} mercadopago {si(r['accepts_mercadopago'])}",
        f"pagos {pays}",
        "etiquetas " + " ".join(r["tags"]),
        "garantia " + (r["warranty"] if isinstance(r["warranty"], str) and r["warranty"].strip() else "no informa"),
    ])

fichas = [norm_text(ficha(r)) for r in both.to_dict("records")]
text = lambda d: fichas[:len(train)] if d is train else fichas[len(train):]
tok = AutoTokenizer.from_pretrained(MODEL)
enc_tr = tok(text(train), truncation=True, max_length=MAXLEN)["input_ids"]
enc_te = tok(text(test), truncation=True, max_length=MAXLEN)["input_ids"]
print(len(enc_tr), len(enc_te), "largo medio en tokens:", np.mean([len(e) for e in enc_tr]).round(1),
      "| truncadas:", np.mean([len(e) >= MAXLEN for e in enc_tr]).round(3))
print(fichas[0])

In [ ]:
def batch(ids_list):
    L = max(len(i) for i in ids_list)
    ids = torch.full((len(ids_list), L), tok.pad_token_id, dtype=torch.long)
    for r, i in enumerate(ids_list):
        ids[r, :len(i)] = torch.tensor(i)
    return ids.to(dev), (ids != tok.pad_token_id).to(dev)

@torch.no_grad()
def predict(model, enc):
    model.eval()
    order = np.argsort([len(e) for e in enc])          # por largo, para rellenar poco
    out = np.zeros(len(enc), dtype="float32")
    for s in range(0, len(enc), 256):
        idx = order[s:s + 256]
        ids, mask = batch([enc[i] for i in idx])
        with torch.autocast("cuda", enabled=dev == "cuda"):
            lg = model(input_ids=ids, attention_mask=mask).logits.float()
        out[idx] = (lg[:, 1] - lg[:, 0]).cpu().numpy()
    return out

def fit_predict(enc_fit, y_fit, enc_pred_list, seed):
    """Entrena desde el modelo preentrenado y devuelve el logit (new - used) para cada conjunto pedido."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(enc_fit) / BS))
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler(enabled=dev == "cuda")
    yt = torch.tensor(y_fit)
    for ep in range(EPOCHS):
        model.train()
        perm = np.random.permutation(len(enc_fit))
        for s in range(0, len(perm), BS):
            idx = perm[s:s + BS]
            ids, mask = batch([enc_fit[i] for i in idx])
            with torch.autocast("cuda", enabled=dev == "cuda"):
                loss = model(input_ids=ids, attention_mask=mask, labels=yt[idx].to(dev)).loss
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
    preds = [predict(model, e) for e in enc_pred_list]
    del model, opt; torch.cuda.empty_cache()
    return preds

In [ ]:
sub = lambda enc, idx: [enc[i] for i in idx]
out = {"oof": np.zeros(len(y), dtype="float32"), "test": np.zeros(len(enc_te), dtype="float32")}
t0 = time.time()
outer = list(StratifiedKFold(5, shuffle=True, random_state=0).split(y, y))      # = features.cv_folds
for f, (a, b) in enumerate(outer):
    enc_a, ya = sub(enc_tr, a), y[a]
    inner = np.zeros(len(a), dtype="float32")
    for k, (i, j) in enumerate(StratifiedKFold(5, shuffle=True, random_state=1).split(ya, ya)):
        (inner[j],) = fit_predict(sub(enc_a, i), ya[i], [sub(enc_a, j)], seed=100 * f + k)
        print(f"fold {f} interno {k}: acc {((inner[j] > 0) == ya[j]).mean():.4f}  ({(time.time() - t0) / 60:.0f} min)", flush=True)
    pb, pt = fit_predict(enc_a, ya, [sub(enc_tr, b), enc_te], seed=100 * f + 9)
    out[f"inner{f}"], out[f"valid{f}"], out[f"test{f}"] = inner, pb, pt
    out["oof"][b] = pb; out["test"] += pt / len(outer)
    print(f"fold {f}: accuracy validación {((pb > 0) == y[b]).mean():.4f}  ({(time.time() - t0) / 60:.0f} min)", flush=True)
    np.savez("bert_level1.npz", **out)
print(f"Accuracy CV solo ficha (BETO): {((out['oof'] > 0) == y).mean():.4f}")